# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This notebook demonstrates how to use Elasticsearch's [MultiQueryRetriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate multiple similar queries for a given user input and retrieve a larger set of relevant documents from a vector store.

**What we will do:**
1. Download a set of fictional workplace documents
2. Split them into chunks and store them in Elasticsearch as vector embeddings
3. Use MultiQueryRetriever + OpenAI to answer questions based on those documents

## 1. Install Packages

In [1]:
# Pin all packages to compatible versions to avoid import conflicts
%pip install -q \
    "langchain==0.2.16" \
    "langchain-core==0.2.38" \
    "langchain-community==0.2.16" \
    "langchain-elasticsearch==0.2.2" \
    "langchain-openai==0.1.23" \
    "langchain-text-splitters==0.2.4" \
    "pydantic==2.8.2" \
    "python-dotenv" \
    "tiktoken" \
    "jq" \
    "lark"



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Import Modules

In [2]:
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_elasticsearch import ElasticsearchStore
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from dotenv import load_dotenv
from urllib.request import urlopen
import logging
import json
import os


## 3. Load API Keys

Create a `.env` file in the same folder as this notebook with the following content:
```
OPENAI_API_KEY=sk-...
ELASTIC_CLOUD_ID=my-deployment:dXMt...
ELASTIC_API_KEY=...
```

In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")

assert OPENAI_API_KEY, "Missing OPENAI_API_KEY in .env"
assert ELASTIC_CLOUD_ID, "Missing ELASTIC_CLOUD_ID in .env"
assert ELASTIC_API_KEY, "Missing ELASTIC_API_KEY in .env"

print("All keys loaded successfully.")


All keys loaded successfully.


## 4. Connect to Elasticsearch

We use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our Elastic Cloud deployment and store document embeddings.

In [4]:
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-documents",
    embedding=embeddings,
)

print("Connected to Elasticsearch.")


Connected to Elasticsearch.


## 5. Download the Dataset

We download a set of fictional workplace documents (JSON format) from the Elastic GitHub repository.

In [5]:
url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as f:
    json.dump(data, f)

print(f"Downloaded {len(data)} documents.")


Downloaded 15 documents.


## 6. Split Documents into Chunks

We split each document into chunks of **800 tokens** with an overlap of **400 tokens**. Overlapping chunks ensure that context is not lost at the boundaries between chunks.

We also extract metadata (name, summary, url, category, updated_at) from each document so it can be displayed alongside retrieved passages.

In [6]:
def metadata_func(record: dict, metadata: dict) -> dict:
    """Extract metadata fields from each JSON record."""
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    # Use None for empty dates — Elasticsearch cannot parse empty datetime strings
    updated_at = record.get("updated_at", "")
    metadata["updated_at"] = updated_at if updated_at else None
    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800,
    chunk_overlap=400,
)

docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Split into {len(docs)} document chunks.")


Split into 15 document chunks.


## 7. Index Documents into Elasticsearch

We embed each chunk using OpenAI and store the vectors in Elasticsearch. We then create the MultiQueryRetriever, which uses an LLM to automatically generate multiple variations of each user question to improve retrieval coverage.

In [7]:
documents = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="workplace-documents",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Log the generated queries so we can see what the retriever produces
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

print("Vectorstore indexed and retriever ready.")


Vectorstore indexed and retriever ready.


## 8. Build the Question-Answering Chain

We build a chain that:
1. Takes a user question
2. Uses MultiQueryRetriever to fetch relevant document chunks
3. Combines those chunks into a context string
4. Passes the context + question to OpenAI for a final answer

In [8]:
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: \"{question}\"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"):
    """Format and combine retrieved documents into a single context string."""
    return document_separator.join(format_document(doc, document_prompt) for doc in docs)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm


## 9. Ask a Question

Let's ask about the NASA sales team. The MultiQueryRetriever will generate several similar questions behind the scenes to maximize relevant results.

In [9]:
ans = chain.invoke("what is the nasa sales team?")
print("---- Answer ----")
print(ans)


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North America South America region, which includes the United States, Canada, Mexico, Central and South America. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


---

## Assignment — Two Creative Iterations

**Generate at least two new iterations of the previous cells. Be creative.** Did you master MultiQueryRetriever concepts through this lab?

### Iteration 1 — Different Topic: Travel & Expense Policy

We reuse the same chain but ask a completely different question. The MultiQueryRetriever will again generate multiple variations of the question to retrieve the most relevant passages from the vector store.

In [10]:
ans2 = chain.invoke("what is the travel and expense policy?")
print("---- Answer ----")
print(ans2)


INFO:langchain.retrievers.multi_query:Generated queries: ["1. Can you provide information on the company's travel and expense policy?", '2. How does the organization handle travel and expense policies?', '3. What are the guidelines for travel and expense reimbursement within the company?']


---- Answer ----

I'm sorry, I cannot answer that question as it is not mentioned in the provided context. This context is about company vacation policy, new employee onboarding, and a work from home policy update. If you have any questions about these policies, I would be happy to assist you.


### Iteration 2 — Custom Prompt: Bullet-Point Answers

We swap in a different prompt template that instructs the model to always respond in structured bullet points. This demonstrates how the same retriever and vector store can produce a completely different output style just by changing the prompt.

In [11]:
BULLET_PROMPT = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Answer the question below using ONLY the provided context.
    Format your answer as a clear, concise bullet-point list.
    If the answer is not in the context, say: 'I could not find this information.'

    Context:
    {context}

    Question: {question}

    Answer (bullet points):
    """
)

bullet_chain = _context | BULLET_PROMPT | llm

ans3 = bullet_chain.invoke("what are the remote work guidelines?")
print("---- Answer (Bullet Points) ----")
print(ans3)


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the guidelines for working remotely?', '2. Can you provide me with the guidelines for remote work?', '3. How can I access the remote work guidelines?']


---- Answer (Bullet Points) ----
- Employees must be eligible for remote work as determined by their role and responsibilities
     - Employees must have approval from their direct supervisor and the HR department
     - Necessary equipment and resources will be provided, including a company-issued laptop and access to secure communication tools
     - Employees are responsible for maintaining and protecting company equipment and data
     - Employees must create a comfortable and safe workspace conducive to productivity
     - Regular communication with supervisors, colleagues, and team members is expected
     - Employees are expected to maintain regular work hours and be available during normal business hours
     - Performance expectations and goals will be established for remote work
     - Accurate time tracking is required, with approval needed for overtime work
     - Adherence to confidentiality and data security policies is required
     - Employees are encouraged to prioriti